## 0. Setup

For Colab, run this cell once. Locally, the course virtual environment already provides the packages and the instructor can set `OXFORD_PET_ROOT` to the cached data directory. The preferred artifacts are a Session 4 checkpoint or saved validation predictions; environment variables `SESSION4_CHECKPOINT` and `SESSION4_PREDICTIONS` can point to them.

In [ ]:
import os
import random
import urllib.request
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from PIL import Image
from torchvision.transforms import functional as TF
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import OxfordIIITPet
from torchvision.transforms import InterpolationMode
from torchvision.models.segmentation import LRASPP_MobileNet_V3_Large_Weights, lraspp_mobilenet_v3_large


SEED = 17
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 16
EPOCHS = 2
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IGNORE_INDEX = 255
CLASS_NAMES = ['background', 'pet']
print('torch:', torch.__version__)

In [ ]:
DATA_ROOT = Path(os.environ.get("OXFORD_PET_ROOT", "data"))
base = OxfordIIITPet(root=DATA_ROOT, split="trainval", target_types="segmentation", download=True)
permutation = torch.randperm(len(base), generator=torch.Generator().manual_seed(SEED)).tolist()
train_indices, val_indices = permutation[:64], permutation[64:88]
assert len(train_indices) == 64 and len(val_indices) == 24
assert not set(train_indices).intersection(val_indices)
print(f"fixed split: train={len(train_indices)}, validation={len(val_indices)}, overlap=0")
print("first five train indices:", train_indices[:5])

In [ ]:
def remap_trimap(raw_mask):
    raw = torch.as_tensor(np.asarray(raw_mask, dtype=np.uint8), dtype=torch.long)
    target = torch.full_like(raw, 255)
    target[raw == 1] = 1  # pet
    target[raw == 2] = 0  # background
    return target

def paired_transform(image, raw_mask):
    image = TF.resize(image, IMAGE_SIZE, interpolation=InterpolationMode.BILINEAR, antialias=True)
    raw_mask = TF.resize(raw_mask, IMAGE_SIZE, interpolation=InterpolationMode.NEAREST)
    image = TF.normalize(TF.to_tensor(image), mean=MEAN, std=STD)
    return image, remap_trimap(raw_mask)

class PetSubset(Dataset):
    def __init__(self, dataset, indices): self.dataset, self.indices = dataset, list(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        image, trimap = self.dataset[self.indices[i]]
        return paired_transform(image, trimap)

train_loader = DataLoader(PetSubset(base, train_indices), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(PetSubset(base, val_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
images, targets = next(iter(train_loader))
assert images.shape == (BATCH_SIZE, 3, 128, 128) and targets.shape == (BATCH_SIZE, 128, 128)
assert set(torch.unique(targets).tolist()) <= {0, 1, 255}
print("batch contract:", tuple(images.shape), tuple(targets.shape), torch.unique(targets).tolist())

## 1. Load the shared baseline (or a transparent fallback)

Preferred order: (1) a saved prediction `.npz`, then (2) a Session 4 LR-ASPP checkpoint. If neither is supplied, the notebook uses a **non-learned RGB heuristic** only so the class can practise evaluation and visualization. That fallback is explicitly not a Session 4 model result and must not be used as a performance claim.

In [ ]:
# 🟩 Corrección JRSA: la notebook de la sesion anterior guarda el checkpoint
# en 'artifacts/session4_baseline.pt' (ARTIFACT_DIR/BASELINE_PATH), no en la
# raiz; con la ruta original esta celda nunca encontraba el archivo. Tambien
# se respeta la variable SESSION4_CHECKPOINT que menciona la celda 0.
checkpoint_path = os.environ.get('SESSION4_CHECKPOINT', 'artifacts/session4_baseline.pt')

model = lraspp_mobilenet_v3_large(weights=None, weights_backbone=None, num_classes=2).eval()
checkpoint = torch.load(checkpoint_path, map_location='cpu')
model.load_state_dict(checkpoint.get('model_state_dict', checkpoint))
with torch.inference_mode():
    logits = model(images)['out']
    probabilities = logits.softmax(dim=1)
predictions, pet_confidence = probabilities.argmax(dim=1), probabilities[:, 1]


assert predictions.shape == targets.shape == pet_confidence.shape
print('Session 4 checkpoint: session4_baseline.pt')

## 2. Exercise E1 — report dataset-level metrics

Complete the confusion-matrix accumulation. Rows are true classes and columns are predicted classes. Accumulate all valid pixels across all 24 examples **before** calculating accuracy and IoU.

In [ ]:
# TODO: return a 2x2 confusion matrix, excluding every target == IGNORE_INDEX


In [ ]:
# 🟩 RESPUESTA (JRSA)
def confusion_matrix_2x2(targets, predictions, num_classes=2):
    """Accumulate a rows=true, columns=predicted confusion matrix over every
    valid (non-ignored) pixel in the batch."""
    confusion = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    valid = targets != IGNORE_INDEX
    true_valid = targets[valid]
    pred_valid = predictions[valid]
    for true_class in range(num_classes):
        for predicted_class in range(num_classes):
            confusion[true_class, predicted_class] = (
                (true_valid == true_class) & (pred_valid == predicted_class)
            ).sum()
    return confusion

confusion = confusion_matrix_2x2(targets, predictions)
total_valid = confusion.sum().item()
pixel_accuracy = confusion.diag().sum().item() / total_valid

tp = confusion[1, 1].item()
fp = confusion[0, 1].item()
fn = confusion[1, 0].item()
pet_iou = tp / (tp + fp + fn) if (tp + fp + fn) else float("nan")

print("confusion matrix (rows=true, cols=predicted):")
print(confusion)
print(f"valid pixels={total_valid} | pixel accuracy={pixel_accuracy:.4f} | pet IoU={pet_iou:.4f}")


The value above is a dataset report: every valid pixel gets one vote. We do **not** average 24 per-image IoUs, because that would weight a tiny pet and a large pet equally.

## 3. Choose three cases by rule

Rule: select the lowest pet-IoU image, then the median and highest pet-IoU images from the remaining examples. This gives one difficult case and two contrasts without cherry-picking by appearance.

In [ ]:
def pet_iou(target, prediction):
    valid = target != IGNORE_INDEX
    truth_pet, predicted_pet = (target == 1) & valid, (prediction == 1) & valid
    union = (truth_pet | predicted_pet).sum().item()
    return (truth_pet & predicted_pet).sum().item() / union if union else np.nan

per_image_iou = np.array([pet_iou(target, prediction) for target, prediction in zip(targets, predictions)])
ordered = np.argsort(np.nan_to_num(per_image_iou, nan=2.0))
low_index = int(ordered[0])
remaining = [int(index) for index in ordered if int(index) != low_index]
chosen_indices = [low_index, remaining[len(remaining) // 2], remaining[-1]]
for index in chosen_indices:
    print(f'pet IoU = {per_image_iou[index]:.3f}')
assert len(set(chosen_indices)) == 3

## 4. Exercise E2 — make aligned visual evidence

Complete the plotting function, then call it for the lowest-pet-IoU case. All five panels use the same `128 × 128` pixel coordinates: original image, ground truth, prediction, pet confidence, and an alpha overlay. Gray ground-truth pixels are ignored borders, not a third class.

In [ ]:
# TODO: complete the five-panel visualization; do not resize panels independently


In [ ]:
# 🟩 RESPUESTA (JRSA)
def image_for_display(image_tensor):
    mean = torch.tensor(MEAN)[:, None, None]
    std = torch.tensor(STD)[:, None, None]
    return (image_tensor.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

MASK_CMAP = ListedColormap(["#183a5a", "#f28e2b", "#9e9e9e"])  # background, pet, ignored border

def mask_for_display(mask_tensor):
    display = mask_tensor.clone()
    display[display == IGNORE_INDEX] = 2
    return display

def show_five_panels(index, alpha=0.5):
    """Five aligned 128x128 panels for one example: image, ground truth,
    prediction, pet confidence, and an alpha overlay of the prediction."""
    image = images[index]
    target = targets[index]
    prediction = predictions[index]
    confidence = pet_confidence[index]

    display_image = image_for_display(image)
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))

    axes[0].imshow(display_image)
    axes[0].set_title("image")

    axes[1].imshow(mask_for_display(target), cmap=MASK_CMAP, vmin=0, vmax=2)
    axes[1].set_title("ground truth")

    axes[2].imshow(mask_for_display(prediction), cmap=MASK_CMAP, vmin=0, vmax=2)
    axes[2].set_title("prediction")

    axes[3].imshow(confidence, cmap="viridis", vmin=0, vmax=1)
    axes[3].set_title("pet confidence")

    axes[4].imshow(display_image)
    axes[4].imshow(prediction, cmap=ListedColormap(["none", "#f28e2b"]), vmin=0, vmax=1, alpha=alpha)
    axes[4].set_title(f"overlay (alpha={alpha})")

    for axis in axes:
        axis.axis("off")
    fig.suptitle(f"index={index} | pet IoU={per_image_iou[index]:.3f}")
    plt.tight_layout()
    plt.show()

show_five_panels(chosen_indices[0], alpha=0.65)


In [ ]:
# Compare the other two rule-selected cases. Changing alpha only changes the display.
show_five_panels(chosen_indices[1], alpha=0.65)
show_five_panels(chosen_indices[2], alpha=0.65)